# 01 Data Exploration

This notebook inspects MovieLens 100K before modeling. The dataset contains ratings, timestamps, user demographics, and movie metadata. It is suitable for this project because it supports collaborative filtering and provides enough side information to study context-aware recommendation.

MovieLens 100K has no real device or session fields. We therefore use available context honestly: time features from timestamps, user demographics, and movie metadata.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data_loader import load_movielens_100k
from src.preprocessing import merge_dataset
from src.analysis import plot_rating_distribution, plot_interactions_by_hour, plot_interactions_by_time_of_day
from src.utils import ensure_dir, set_seed

set_seed()
ensure_dir(config.FIGURES_DIR)

## Load Raw Data

If this cell fails, place `u.data`, `u.user`, and `u.item` under `data/raw/ml-100k/`.

In [ ]:
ratings, users, movies = load_movielens_100k(config.RAW_DATA_DIR)
print('ratings:', ratings.shape)
print('users:', users.shape)
print('movies:', movies.shape)
display(ratings.head())
display(users.head())
display(movies.head())

In [ ]:
print('Number of users:', ratings['user_id'].nunique())
print('Number of movies:', ratings['movie_id'].nunique())
print('Number of ratings:', len(ratings))
print('\nRatings per user:')
display(ratings.groupby('user_id').size().describe())
print('\nRatings per movie:')
display(ratings.groupby('movie_id').size().describe())

## Context Features

Timestamps provide hour, day of week, month, weekend indicator, and time-of-day buckets. User metadata provides age, gender, and occupation. Movie metadata provides genres and release year.

In [ ]:
merged = merge_dataset(ratings, users, movies)
display(merged[['user_id', 'movie_id', 'rating', 'datetime', 'hour', 'day_of_week', 'is_weekend', 'month', 'time_of_day', 'age_group', 'release_year']].head())

In [ ]:
plot_rating_distribution(merged)
plot_interactions_by_hour(merged)
plot_interactions_by_time_of_day(merged)
print('Saved exploration plots to', config.FIGURES_DIR)

In [ ]:
genre_counts = movies[config.GENRE_COLUMNS].sum().sort_values(ascending=False)
display(genre_counts)
genre_counts.plot(kind='bar', figsize=(10, 4), color='#4C78A8', title='Movie Genre Distribution')
plt.ylabel('Movies')
plt.tight_layout()
plt.show()

## Summary

The dataset supports a controlled context-aware recommendation study. The main missing context is real session/device data, so the project avoids inventing it and focuses on temporal, user, and movie context.